In [1]:
!pip3 install trimesh vtk networkx

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121


In [ ]:
import vtk
import numpy as np
from vtk.util import numpy_support
import networkx as nx
from scipy.sparse import csr_matrix

def split_mesh_by_sharp_edges(graph, flags, min_graph_size):

    # remove the nodes whose flags are 1
    nodes_to_remove = []
    for node in graph.nodes():
        if flags[node] == 1:
            nodes_to_remove.append(node)
    new_graph = graph.copy()
    new_graph.remove_nodes_from(nodes_to_remove)

    # find the seperated components and keep the large ones
    # return a list of sets of nodes
    all_graph_nodes = list(nx.connected_components(new_graph))  
    
    # if these nodes are just make the graph smaller but not split the graph
    if len(all_graph_nodes) == 1:
        return [graph]
    else:
        all_large_graphs = []
        for i in range(len(all_graph_nodes)):
            nodes = all_graph_nodes[i]
            if len(nodes) > min_graph_size:
                splited_subgraph = graph.subgraph(nodes).copy()
                all_large_graphs.append(splited_subgraph)
        # if all split graphs are too small, return the original graph
        if len(all_large_graphs) <= 1:
            return [graph]
        else:
            return all_large_graphs

def mark_sharp_edges(polydata, angle_threshold=20.0):
    poly = polydata  # your vtkPolyData surface
    angle_deg = angle_threshold  # threshold; increase for noisier meshes

    # (Optional) keep a mapping back to original point IDs
    idf = vtk.vtkIdFilter()
    idf.SetInputData(poly)
    idf.SetPointIds(True)
    idf.PointIdsOn()
    idf.SetPointIdsArrayName("OrigPointIds")  # Fixed method name
    idf.Update()

    fe = vtk.vtkFeatureEdges()
    fe.SetInputConnection(idf.GetOutputPort())
    fe.FeatureEdgesOn()
    fe.BoundaryEdgesOff()     # set On() if you also want open borders
    fe.NonManifoldEdgesOn()
    fe.SetFeatureAngle(angle_deg)
    fe.Update()

    edges_poly = fe.GetOutput()

    # Build (N,1) flag on original vertices
    N = poly.GetNumberOfPoints()
    flags = np.zeros((N, 1), dtype=np.uint8)

    # Use preserved original point IDs from the IdFilter
    orig_ids = numpy_support.vtk_to_numpy(edges_poly.GetPointData().GetArray("OrigPointIds"))

    # Each line cell references two points from edges_poly; map those points back
    lines = edges_poly.GetLines()
    idlist = vtk.vtkIdList()
    lines.InitTraversal()
    while lines.GetNextCell(idlist):
        for j in range(idlist.GetNumberOfIds()):
            pid_edgepoly = idlist.GetId(j)
            pid_orig = int(orig_ids[pid_edgepoly])
            flags[pid_orig, 0] = 1

    print(f"Feature edges detected: {np.sum(flags)} points marked as sharp edges")
    print(f"Sharp edge ratio: {np.sum(flags) / N * 100:.2f}%")

    return flags

def find_nearest_neighbors_from_mesh(polydata):

    # Extract cell connectivity
    max_node_id = 0
    cells = polydata.GetPolys()
    num_nodes = polydata.GetNumberOfPoints()
    if cells:
        # Get the connectivity array
        connectivity = cells.GetData()
        connectivity_array = numpy_support.vtk_to_numpy(connectivity)
        
        # Get cell information
        num_cells = polydata.GetNumberOfCells()
        
        # Extract edges from polygon connectivity
        nearest_neighbors = [[] for _ in range(num_nodes)]
        
        idx = 0
        for i in range(num_cells):
            num_vertices = connectivity_array[idx]
            vertices = connectivity_array[idx+1:idx+1+num_vertices]

            vertices = vertices.tolist()
            vertices = vertices + [vertices[0]]

            for j in range(num_vertices):
                nearest_neighbors[vertices[j]].append(vertices[(j+1)])
                nearest_neighbors[vertices[(j+1)]].append(vertices[j])

            max_node_id = max(max_node_id, max(vertices))
            
            idx += (1 + num_vertices)

    for i in range(len(nearest_neighbors)):
        nearest_neighbors[i] = list(set(nearest_neighbors[i]))
    
    print('max node id: ', max_node_id)

    return nearest_neighbors

def find_clusters_from_mesh(coords, org_graph):

    num_nodes = len(org_graph.nodes())
    all_graph_nodes = list(nx.connected_components(org_graph))  # List of sets of nodes
    all_graph_nodes = [np.array(list(comp)) for comp in all_graph_nodes]
    print(f"Number of connected components: {len(all_graph_nodes)}")
    print(f"Component sizes: {[len(comp) for comp in all_graph_nodes]}")

    '''
    classify the graph nodes into different clusters
        - the largest cluster is the vehicle surface -> cluster #1
        - 8 second largest cluster are the vehicle wheels
    '''
    flags = np.zeros(num_nodes)
    # Get the vehicle surface
    graph_sizes = [len(comp) for comp in all_graph_nodes]
    surface_index = np.argmax(graph_sizes)
    flags[all_graph_nodes[surface_index]] = 1

    # Get the vehicle wheels
    X_bar = 1.5
    Y_bar = 0.0
    size_orders = np.argsort(graph_sizes)[::-1]
    size_orders = size_orders[1:9]
    for index in size_orders:
        coors = coords[all_graph_nodes[index]]
        center = np.mean(coors, axis=0)
        # W1
        if center[0] > X_bar and center[1] > Y_bar:
            flags[all_graph_nodes[index]] = 2
        # W2
        elif center[0] > X_bar and center[1] < Y_bar:
            flags[all_graph_nodes[index]] = 3
        # W3
        elif center[0] < X_bar and center[1] > Y_bar:
            flags[all_graph_nodes[index]] = 4
        # W4
        elif center[0] < X_bar and center[1] < Y_bar:
            flags[all_graph_nodes[index]] = 5
    
    # for all the other clusters, set the flag to 6
    flags[flags == 0] = 6
    
    return flags


def create_seg_matrix(coords, polydata, threshold_angles, 
    min_graph_size_ratio = 0.0001, FIND_SEG = True):

    # extract basic information
    num_nodes = len(coords)

    # create neighbor information
    neighbor_indices = find_nearest_neighbors_from_mesh(polydata)

    # create a graph
    org_graph = nx.Graph()
    org_graph.add_nodes_from(range(len(coords)))
    for i in range(len(coords)):
        for j in neighbor_indices[i]:
            org_graph.add_edge(i, j)

    '''
    find_clusters_from_mesh(coords, org_graph)
    '''
    node_cluster_flags = find_clusters_from_mesh(coords, org_graph)

    if FIND_SEG:
        '''
        The graph is already a combination of connected components
        Only the main surface is changing, so we only need to extract physics tokens from it
        '''
        graph_nodes = list(nx.connected_components(org_graph))  # Convert generator to list
        print(f"Number of connected components: {len(graph_nodes)}")
        print(f"Component sizes: {[len(comp) for comp in graph_nodes]}")

        # Convert sets of nodes into actual NetworkX Graph objects
        # find the largest node set and only care about it
        largest_node_set = max(graph_nodes, key=len)
        subgraph = org_graph.subgraph(largest_node_set).copy()
        graph_still_need_split = [subgraph]

        # iteratively split the mesh based on the sharp edges
        # threshold_angles = [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
        all_flags = []
        all_sharp_edges = []
        identified_sharp_edges_last_iteration = set([])
        sharp_edge_subgraphs = []  # Store subgraphs created from newly detected sharp edges
        
        for threshold_angle in threshold_angles:
            print(f"Processing threshold angle: {threshold_angle}")
            flags = mark_sharp_edges(polydata, angle_threshold=threshold_angle)
            all_flags.append(flags)

            # store the identified sharp edges in this iteration
            sharp_edges_this_threshold = []
            for i in range(len(flags)):
                if flags[i] == 1:
                    sharp_edges_this_threshold.append(i)
            sharp_edges_this_threshold = set(sharp_edges_this_threshold)
            
            # not include the sharp edges that are already in the all_sharp_edges
            sharp_edges_this_threshold_after_pruning = sharp_edges_this_threshold - identified_sharp_edges_last_iteration
            all_sharp_edges.append(list(sharp_edges_this_threshold_after_pruning))
            
            # Create subgraphs from newly detected sharp edge nodes
            if len(sharp_edges_this_threshold_after_pruning) > 0:
                # Create a subgraph from the original graph containing only the newly detected sharp edge nodes
                sharp_edge_subgraph = org_graph.subgraph(list(sharp_edges_this_threshold_after_pruning))
                # Find connected components in this subgraph
                connected_components = list(nx.connected_components(sharp_edge_subgraph))
                # Add each connected component as a subgraph
                for component in connected_components:
                    if len(component) > 0:
                        component_subgraph = org_graph.subgraph(component).copy()
                        sharp_edge_subgraphs.append(component_subgraph)
                print(f"Found {len(connected_components)} connected components from newly detected sharp edges")
            
            identified_sharp_edges_last_iteration = sharp_edges_this_threshold
            
            # seperate the mesh by the sharp edges
            graph_still_need_split_this_iteration = []
            for graph in graph_still_need_split:
                result = split_mesh_by_sharp_edges(graph, flags, min_graph_size=int(min_graph_size_ratio * num_nodes))
                graph_still_need_split_this_iteration.extend(result)
            assert len(graph_still_need_split_this_iteration) >= len(graph_still_need_split), '# of split graphs should not be decreasing'
            
            # update the graph_still_need_split
            graph_still_need_split = graph_still_need_split_this_iteration
            print('current number of subgraphs: ', len(graph_still_need_split))
            if len(graph_still_need_split) == 0:
                break

        # combine the subgraphs from mesh splitting and sharp edge detection
        all_subgraphs = graph_still_need_split + sharp_edge_subgraphs
        print('number of subgraphs before deduplication: ', len(all_subgraphs))
        print(f'  - {len(graph_still_need_split)} from mesh splitting')
        print(f'  - {len(sharp_edge_subgraphs)} from sharp edge detection')
        
        # Remove duplicate subgraphs based on node sets (find unique separated graphs)
        unique_subgraphs = []
        seen_node_sets = set()
        for subgraph in all_subgraphs:
            node_set = frozenset(subgraph.nodes())
            if node_set not in seen_node_sets:
                seen_node_sets.add(node_set)
                unique_subgraphs.append(subgraph)
        
        all_subgraphs = unique_subgraphs
        print('number of subgraphs after deduplication: ', len(all_subgraphs))

        # sort the subgraphs by the size
        all_subgraphs = sorted(all_subgraphs, key=lambda x: len(x), reverse=True)

        # create the seg matrix as a sparse matrix
        # Prepare data for sparse matrix construction
        row_indices = []
        col_indices = []
        data = []
        
        for i in range(len(all_subgraphs)):
            subgraph_size = len(all_subgraphs[i])
            value = 1 / subgraph_size
            for node in all_subgraphs[i]:
                row_indices.append(i)
                col_indices.append(node)
                data.append(value)
        
        # Create sparse matrix in CSR format
        seg_matrix = csr_matrix((data, (row_indices, col_indices)), 
                            shape=(len(all_subgraphs), len(coords)))
    
    else:
        seg_matrix = np.ones((1, 1, 1))

    return seg_matrix, node_cluster_flags

In [3]:
import os
import pandas as pd
import h5py
import re
import trimesh

def read_dat_file(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
        # 找 "N = {N}"的内容
        pattern = re.compile(r'N = (\d+)')
        N = int(pattern.findall(lines[2])[0])
        points = lines[3:3+N]
        mesh = lines[3+N:]
        pos_list = []
        value_list = []
        for d in points:
            d = d.strip().split()
            pos_list.append([float(x.strip()) for x in d[:3]])
            value_list.append([float(x.strip()) for x in d[3:]])
        mesh_list = []
        for m in mesh:
            m = m.strip().split()
            m = [float(x.strip()) - 1 for x in m]
            mesh_list.append(m)
        cur_mesh = trimesh.Trimesh(vertices=pos_list, faces=mesh_list)
        normals = cur_mesh.vertex_normals
        edges = cur_mesh.edges_unique

        # create seg matrix
        coords = np.array(pos_list, dtype=np.float64)
        
        # Create VTK points
        vtk_points = vtk.vtkPoints()
        vtk_points.SetData(numpy_support.numpy_to_vtk(coords))
        
        # Create VTK cells (faces)
        vtk_cells = vtk.vtkCellArray()
        for face in mesh_list:
            face_array = np.array(face, dtype=np.int64)
            vtk_cells.InsertNextCell(len(face_array))
            for vertex_id in face_array:
                vtk_cells.InsertCellPoint(int(vertex_id))
        
        # Create polydata
        polydata = vtk.vtkPolyData()
        polydata.SetPoints(vtk_points)
        polydata.SetPolys(vtk_cells)
        
        # Call create_seg_matrix (threshold_angles parameter is ignored, using default internal values)
        seg_matrix, _ = create_seg_matrix(coords, polydata, threshold_angles=np.arange(1.0, 0.1, -0.2))
        # Convert sparse matrix to dense array for h5py storage
        seg_matrix_dense = seg_matrix.toarray()

    return pos_list, value_list, mesh_list, normals, edges, seg_matrix_dense

# data = read_dat_file("/path/to/data")
# data

In [4]:
# 读取 xlsx 文件
def read_xlsx_file(file_path):
    data = pd.read_csv(file_path)
    return data
csv = read_xlsx_file("/taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/result_new.csv")
csv

,idx,Ma,alpha,beta,CA,CN,CZ,Cl,Cn,Cm
0,1,2,0,0,0.380464,-0.378518,0.000000,0.000000,0.000000,-0.036830
1,1,7,0,0,0.096002,-0.252061,0.000000,0.000000,0.000000,0.002656
2,1,7,0,2,0.096458,-0.253052,-0.046005,0.001555,-0.027887,0.002468
3,1,7,7,0,0.072269,0.487737,0.000000,0.000000,0.000000,-0.007683
4,1,7,7,2,0.072440,0.486681,-0.033188,-0.003362,-0.020575,-0.007585
...,...,...,...,...,...,...,...,...,...,...
115,73,2,0,0,0.569820,-1.189360,0.000000,0.000000,0.000000,0.259080
116,73,7,0,0,0.155642,-0.528201,0.000000,0.000000,0.000000,0.100559
117,73,7,0,2,0.155734,-0.527520,-0.050453,0.003987,-0.032567,0.099887
118,73,7,7,0,0.086188,0.299927,0.000000,0.000000,0.000000,0.086201


In [5]:
import numpy as np
# 获取当前文件夹下所有文件，提取倒数第二层的数字和倒数第一层的文件夹名
def get_files(root_dir):
    files = []
    for root, dirs, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith('.dat'):
                # 倒数第二层的数字
                num = int(root.split('/')[-2])
                # 倒数第一层的文件夹名
                folder = root.split('/')[-1]
                params = [float(f[-4:]) for f in folder.split('_')]
                params.insert(0, num)
                print(params)
                # 搜索csv 中 idx=num, Ma=params[1], alpha=params[2], beta=params[3]的数据
                idx = csv[(csv['idx'] == num) & (csv['Ma'] == params[1]) & (csv['alpha'] == params[2]) & (csv['beta'] == params[3])]
                if len(idx) == 0:
                    print(f"idx={num}, Ma={params[1]}, alpha={params[2]}, beta={params[3]} not found")
                    continue
                # Check if output file already exists
                output_file = f"/taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/processed_data2/{num}_{params[1]}_{params[2]}_{params[3]}.h5"
                if os.path.exists(output_file):
                    print(f"File already exists, skipping: {output_file}")
                    continue
                data = read_dat_file(os.path.join(root, filename))
                
                with h5py.File(output_file, 'w') as hf:
                    hf.create_dataset("pos", data=data[0])
                    hf.create_dataset("values", data=data[1])
                    hf.create_dataset("normals", data=data[3])
                    hf.create_dataset("seg_matrix", data=data[-1])
                    # attributes
                    hf.attrs['Ma'] = params[1]
                    hf.attrs['alpha'] = params[2]
                    hf.attrs['beta'] = params[3]
                    hf.attrs['CA'] = idx['CA'].values[0]
                    hf.attrs['CN'] = idx['CN'].values[0]
                    hf.attrs['CZ'] = idx['CZ'].values[0]
                    hf.attrs['Cl'] = idx['Cl'].values[0]
                    hf.attrs['Cm'] = idx['Cm'].values[0]
                    hf.attrs['Cn'] = idx['Cn'].values[0]
                print(data[-1].shape)
                break
                with h5py.File(f"/taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/processed_data2/{num}_{params[1]}_{params[2]}_{params[3]}_edge.h5", 'w') as hf:
                    hf.create_dataset("edges", data=data[-1])
    return files

get_files("/taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/")

[33, 7.0, 0.0, 0.0]
File already exists, skipping: /taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/processed_data2/33_7.0_0.0_0.0.h5
[33, 7.0, 0.0, 2.0]
File already exists, skipping: /taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/processed_data2/33_7.0_0.0_2.0.h5
[33, 2.0, 0.0, 0.0]
File already exists, skipping: /taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/processed_data2/33_2.0_0.0_0.0.h5
[33, 7.0, 7.0, 0.0]
File already exists, skipping: /taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/processed_data2/33_7.0_7.0_0.0.h5
[33, 7.0, 7.0, 2.0]
File already exists, skipping: /taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/processed_data2/33_7.0_7.0_2.0.h5
[2, 7.0, 0.0, 0.0]
File already exists, skipping: /taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/processed_data2/2_7.0_0.0_0.0.h5
[2, 7.0, 0.0, 2.0]
File already exists, skipping: /taiga/illinois/eng/cee/meidani/Vincent/aircraft_industry/processed_data2/2_7.0_0.0_2.0.h5
[2,

/tmp/ipykernel_1696719/4051970516.py:42: DeprecationWarning: Call to deprecated class vtkIdFilter. (Please use `vtkGenerateIds` instead.) -- Deprecated since version 9.4.0.
  idf = vtk.vtkIdFilter()


Feature edges detected: 63978 points marked as sharp edges
Sharp edge ratio: 19.27%
current number of subgraphs:  9
Processing threshold angle: 0.8
Feature edges detected: 70373 points marked as sharp edges
Sharp edge ratio: 21.20%
current number of subgraphs:  10
Processing threshold angle: 0.6000000000000001
Feature edges detected: 115741 points marked as sharp edges
Sharp edge ratio: 34.86%
current number of subgraphs:  27
Processing threshold angle: 0.40000000000000013
Feature edges detected: 171100 points marked as sharp edges
Sharp edge ratio: 51.54%
current number of subgraphs:  78
Processing threshold angle: 0.20000000000000018
Feature edges detected: 278327 points marked as sharp edges
Sharp edge ratio: 83.84%
current number of subgraphs:  115
number of subgraphs before physics token purning:  115
(115, 331971)
[42, 7.0, 0.0, 2.0]
max node id:  331970
Number of connected components: 1
Component sizes: [331971]
Number of connected components: 1
Component sizes: [331971]
Process

[]